<a href="https://colab.research.google.com/github/kuds/rl-connect-four/blob/main/%5BConnect%20Four%5D%20DQN%20Self%20Play.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Playing Connect Four with DQN Self Play

Mirrors the PPO notebook in this repo, but trains DQN against the same
`SelfPlayCallback` league. Writes its agent under
`$ARTIFACT_DIR/agents/dqn/` so the tournament notebook can pick it up
alongside PPO / AlphaZero.

## References:
- [Ray's rllib - Documentation](https://docs.ray.io/en/latest/rllib/index.html)
- [Ray's rllib - Multi-Agent Connect 4](https://github.com/ray-project/ray/blob/master/rllib/examples/multi_agent/self_play_with_open_spiel.py)
- [OpenSpiel - Documentation](https://openspiel.readthedocs.io/en/latest/intro.html)


In [ ]:
!pip install ray[rllib]

In [ ]:
!pip install gputil open_spiel gymnasium "pettingzoo[classic]"


In [ ]:
import functools
import json
import math
import multiprocessing as mp
import platform
import shutil
from importlib.metadata import version
from pathlib import Path

import imageio
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ray
import torch
from ray.air.constants import TRAINING_ITERATION
from ray.rllib.core.rl_module.default_model_config import DefaultModelConfig
from ray.rllib.core.rl_module.multi_rl_module import MultiRLModuleSpec
from ray.rllib.core.rl_module.rl_module import RLModuleSpec
from ray.rllib.env.utils import try_import_open_spiel, try_import_pyspiel
from ray.rllib.env.wrappers.open_spiel import OpenSpielEnv
from ray.rllib.examples.multi_agent.utils import (
    ask_user_for_action,
    SelfPlayCallback,
)
from ray.rllib.examples.rl_modules.classes.random_rlm import RandomRLModule
from ray.rllib.utils.metrics import (
    ENV_RUNNER_RESULTS,
    NUM_ENV_STEPS_SAMPLED_LIFETIME,
)
from ray.rllib.utils.test_utils import run_rllib_example_script_experiment
from ray.tune.registry import get_trainable_cls, register_env


In [ ]:
print(f"Python Version: {platform.python_version()}")
print(f"Torch Version: {version('torch')}")
print(f"Is Cuda Available: {torch.cuda.is_available()}")
print(f"Cuda Version: {torch.version.cuda}")
print(f"Numpy Version: {version('numpy')}")
print(f"Ray Version: {version('ray')}")
print(f"Gymnasium Version: {version('Gymnasium')}")
print(f"Open Spiel Version: {version('open_spiel')}")

In [ ]:
print(f"Number or CPUs Available: {mp.cpu_count()}")

In [ ]:
open_spiel = try_import_open_spiel(error=True)
pyspiel = try_import_pyspiel(error=True)

# Import after try_import_open_spiel, so we can error out with hints.
from open_spiel.python.rl_environment import Environment  # noqa: E402

In [ ]:
class Args:
    def __init__(self):
        self.env = "connect_four"
        self.checkpoint_freq = 1
        self.checkpoint_at_end = True
        self.win_rate_threshold = 0.95
        self.min_league_size = 3
        # Default to headless so the notebook can run end-to-end without
        # stdin. Set to > 0 to play against the trained agent interactively.
        self.num_episodes_human_play = 0
        self.from_checkpoint = None
        self.algo = "DQN"
        self.num_env_runners = 2
        self.enable_new_api_stack = None
        self.stop_timesteps = 2000000
        self.stop_iters = 100
        self.as_release_test = False
        self.num_cpus = 10
        self.local_mode = False
        self.old_api_stack = False
        self.framework = "torch"
        self.num_gpus_per_learner = 1
        self.num_learners = 1
        self.evaluation_interval = 0
        self.log_level = None
        self.output = None
        self.no_tune = False
        self.num_agents = 0
        self.num_aggregator_actors_per_learner = None
        self.num_cpus_per_learner = None
        self.verbose = 2
        self.num_samples = 1
        self.max_concurrent_trials = None
        self.as_test = False
        self.num_envs_per_env_runner = 1


args = Args()


In [ ]:
register_env(
    "open_spiel_env",
    lambda _: OpenSpielEnv(pyspiel.load_game(args.env)),
)


def agent_to_module_mapping_fn(agent_id, episode, **kwargs):
    # agent_id = [0|1] -> module depends on episode ID
    # This way, we make sure that both modules sometimes play agent0
    # (start player) and sometimes agent1 (player to move 2nd).
    return "main" if hash(episode.id_) % 2 == agent_id else "random"


config = (
    get_trainable_cls(args.algo)
    .get_default_config()
    .environment("open_spiel_env")
    .callbacks(
        functools.partial(
            SelfPlayCallback,
            win_rate_threshold=args.win_rate_threshold,
        )
    )
    .env_runners(
        num_env_runners=(args.num_env_runners or 2),
        num_envs_per_env_runner=1,
    )
    .multi_agent(
        policies={"main", "random"},
        policy_mapping_fn=agent_to_module_mapping_fn,
        policies_to_train=["main"],
    )
    .rl_module(
        model_config=DefaultModelConfig(fcnet_hiddens=[512, 512]),
        rl_module_spec=MultiRLModuleSpec(
            rl_module_specs={
                "main": RLModuleSpec(),
                "random": RLModuleSpec(module_class=RandomRLModule),
            }
        ),
    )
)

# DQN-specific knobs. RLlib's defaults work; these tweaks are
# deliberately small to keep the notebook close to PPO and finish in
# similar wall-clock on Colab. Tune up for stronger play (larger train
# batch, longer training, custom replay buffer).
if args.algo == "DQN":
    config = config.training(
        train_batch_size=512,
        lr=5e-4,
    )

stop = {
    NUM_ENV_STEPS_SAMPLED_LIFETIME: args.stop_timesteps,
    TRAINING_ITERATION: args.stop_iters,
    "league_size": args.min_league_size,
}


In [ ]:
# Train the "main" policy to play really well using self-play.
results = None
if not args.from_checkpoint:
    results = run_rllib_example_script_experiment(
        config, args, stop=stop
    )

## Post-training artifacts

The cells below are evaluated automatically when the notebook is run
end-to-end. They:

1. Rebuild the trained algorithm from the final checkpoint (with every
   league snapshot the `SelfPlayCallback` added during training).
2. Plot the learning curves that drove the README numbers.
3. Quantitatively evaluate `main` vs a random baseline and vs every
   league snapshot.
4. Persist a stable copy of the checkpoint and a JSON results summary to
   `./artifacts/` so nothing has to be hand-typed into the README.


### Mount Google Drive


In [ ]:
# Mount Google Drive so models, learning-curve plots, game videos, and the
# results summary persist across Colab runtimes. Set USE_GDRIVE = False to
# keep everything local under ./artifacts/ (e.g. when running outside
# Colab).
USE_GDRIVE = True
GDRIVE_ARTIFACT_SUBDIR = "rl-connect-four"

drive_mounted = False
if USE_GDRIVE:
    try:
        from google.colab import drive  # noqa: WPS433 (Colab-only import)
        drive.mount("/content/drive", force_remount=False)
        drive_mounted = True
    except (ImportError, ModuleNotFoundError):
        print("Not running in Colab — Google Drive unavailable, using ./artifacts/")
    except Exception as e:
        print(f"Could not mount Google Drive ({e!r}); using ./artifacts/")

if drive_mounted:
    ARTIFACT_DIR = Path("/content/drive/MyDrive") / GDRIVE_ARTIFACT_SUBDIR
else:
    ARTIFACT_DIR = Path("artifacts")

# Per-agent directory so the tournament notebook can discover this agent
# alongside other algorithms via artifacts/agents/*/. Default name is the
# lowercased algo, override with AGENT_NAME if you want to keep multiple
# runs of the same algo side by side.
AGENT_NAME = args.algo.lower()
AGENT_DIR = ARTIFACT_DIR / "agents" / AGENT_NAME
VIDEO_DIR = AGENT_DIR / "videos"

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
AGENT_DIR.mkdir(parents=True, exist_ok=True)
VIDEO_DIR.mkdir(parents=True, exist_ok=True)
print(f"Artifact directory: {ARTIFACT_DIR}")
print(f"Agent directory:    {AGENT_DIR}")
print(f"Video directory:    {VIDEO_DIR}")


In [ ]:
# Rebuild the trained algorithm from the final checkpoint. We use
# Algorithm.from_checkpoint (not config.build() + restore) because the
# SelfPlayCallback dynamically adds `main_v0`, `main_v1`, ... modules to
# the algorithm during training; from_checkpoint restores the full module
# structure exactly as it was at save time.
from ray.rllib.algorithms.algorithm import Algorithm

# Use the final iteration's checkpoint (training stops on league_size, not
# on a maximized metric, so "best" == "last" here — be explicit about it).
final_result = results.get_best_result(
    metric=TRAINING_ITERATION, mode="max"
)
final_checkpoint = final_result.checkpoint
if final_checkpoint is None:
    raise RuntimeError("Training finished without producing a checkpoint.")

eval_algo = Algorithm.from_checkpoint(str(Path(final_checkpoint.path)))
main_module = eval_algo.get_module("main")

# Discover league snapshots the callback added during training. The
# callback names them main_v0, main_v1, ... so we probe until one is
# missing.
snapshot_modules = {}
idx = 0
while True:
    name = f"main_v{idx}"
    try:
        snapshot_modules[name] = eval_algo.get_module(name)
    except (KeyError, ValueError):
        break
    idx += 1

print(f"Loaded main + {len(snapshot_modules)} league snapshot(s): "
      f"{list(snapshot_modules.keys())}")


In [ ]:
# Evaluation helpers. Every policy callable has signature
# (obs: np.ndarray, legal_actions: list[int]) -> int so we can mix and
# match learned policies with a random baseline below.


def greedy_action(module, obs, legal_actions):
    """Greedy action from an RLModule, masking illegal moves before argmax."""
    with torch.no_grad():
        out = module.forward_inference(
            {"obs": torch.from_numpy(obs).unsqueeze(0)}
        )
    logits = out["action_dist_inputs"][0].detach().cpu().numpy()
    masked = np.full_like(logits, -np.inf, dtype=np.float32)
    masked[legal_actions] = logits[legal_actions]
    return int(np.argmax(masked))


def random_action(obs, legal_actions):
    return int(np.random.choice(legal_actions))


def play_match(env, agent_p0, agent_p1):
    """Play one game; return the (p0_reward, p1_reward) list from OpenSpiel."""
    time_step = env.reset()
    while not time_step.last():
        pid = time_step.observations["current_player"]
        obs = np.asarray(
            time_step.observations["info_state"][pid], dtype=np.float32,
        )
        legal = time_step.observations["legal_actions"][pid]
        action = (agent_p0 if pid == 0 else agent_p1)(obs, legal)
        time_step = env.step([action])
    return time_step.rewards


def evaluate(main_fn, opp_fn, num_games_per_side, env_name):
    """Play num_games_per_side as player 0 and as player 1 against opp_fn."""
    rates = {}
    for role, key in [(0, "main_as_p0"), (1, "main_as_p1")]:
        env = Environment(env_name)
        wins = losses = draws = 0
        for _ in range(num_games_per_side):
            if role == 0:
                rewards = play_match(env, main_fn, opp_fn)
            else:
                rewards = play_match(env, opp_fn, main_fn)
            main_reward = rewards[role]
            if main_reward > 0:
                wins += 1
            elif main_reward < 0:
                losses += 1
            else:
                draws += 1
        n = num_games_per_side
        rates[key] = {
            "wins": wins,
            "losses": losses,
            "draws": draws,
            "win_rate": wins / n,
            "loss_rate": losses / n,
            "draw_rate": draws / n,
            "games": n,
        }
    return rates


def overall_win_rate_ci(eval_result):
    """Aggregate win rate across both sides, with a 95% normal-approx CI."""
    wins = sum(side["wins"] for side in eval_result.values())
    n = sum(side["games"] for side in eval_result.values())
    p = wins / n if n else 0.0
    se = math.sqrt(p * (1 - p) / n) if n else 0.0
    return {
        "win_rate": p,
        "ci95_low": p - 1.96 * se,
        "ci95_high": p + 1.96 * se,
        "wins": wins,
        "games": n,
    }


### Learning curves


In [ ]:
# Learning curves from Tune's per-iteration metrics. Falls back to
# progress.csv when Result.metrics_dataframe is not available on older
# Ray versions.
try:
    metrics_df = final_result.metrics_dataframe
    if metrics_df is None:
        raise AttributeError
except AttributeError:
    metrics_df = pd.read_csv(Path(final_result.path) / "progress.csv")


def first_existing(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


# Env-runner rollouts.
win_rate_col = first_existing(metrics_df, [
    f"{ENV_RUNNER_RESULTS}/win_rate",
    "env_runners/win_rate",
])
return_col = first_existing(metrics_df, [
    f"{ENV_RUNNER_RESULTS}/episode_return_mean",
    "env_runners/episode_return_mean",
])
# Learner-side losses (new-api-stack keys; fall back to old names).
policy_loss_col = first_existing(metrics_df, [
    "learners/main/policy_loss",
    "info/learner/main/learner_stats/policy_loss",
])
vf_loss_col = first_existing(metrics_df, [
    "learners/main/vf_loss",
    "info/learner/main/learner_stats/vf_loss",
])
entropy_col = first_existing(metrics_df, [
    "learners/main/entropy",
    "info/learner/main/learner_stats/entropy",
])
league_col = "league_size" if "league_size" in metrics_df.columns else None

x = metrics_df["training_iteration"]

# --- Combined 2x3 panel ---------------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

# (0) Win rate
if win_rate_col is not None:
    axes[0].plot(x, metrics_df[win_rate_col], color="#2a9d8f")
    axes[0].axhline(
        args.win_rate_threshold, color="red", linestyle="--",
        label=f"threshold ({args.win_rate_threshold})",
    )
    axes[0].legend(loc="lower right")
axes[0].set_title("Win rate (main vs current opponent)")
axes[0].set_xlabel("training iteration")
axes[0].set_ylabel("win rate")
axes[0].set_ylim(-0.05, 1.05)
axes[0].grid(alpha=0.3)

# (1) Episode return / reward
if return_col is not None:
    axes[1].plot(x, metrics_df[return_col], color="#e76f51")
axes[1].set_title("Episode return (reward per episode)")
axes[1].set_xlabel("training iteration")
axes[1].set_ylabel("episode return")
axes[1].grid(alpha=0.3)

# (2) League size
if league_col is not None:
    axes[2].step(x, metrics_df[league_col], where="post", color="#264653")
axes[2].set_title("League size")
axes[2].set_xlabel("training iteration")
axes[2].set_ylabel("league size")
axes[2].grid(alpha=0.3)

# (3) Policy loss
if policy_loss_col is not None:
    axes[3].plot(x, metrics_df[policy_loss_col], color="#6a4c93")
    axes[3].set_title("Policy loss")
else:
    axes[3].set_title("Policy loss (not logged)")
axes[3].set_xlabel("training iteration")
axes[3].set_ylabel("loss")
axes[3].grid(alpha=0.3)

# (4) Value loss
if vf_loss_col is not None:
    axes[4].plot(x, metrics_df[vf_loss_col], color="#f4a261")
    axes[4].set_title("Value function loss")
else:
    axes[4].set_title("Value function loss (not logged)")
axes[4].set_xlabel("training iteration")
axes[4].set_ylabel("loss")
axes[4].grid(alpha=0.3)

# (5) Entropy
if entropy_col is not None:
    axes[5].plot(x, metrics_df[entropy_col], color="#1d3557")
    axes[5].set_title("Policy entropy")
else:
    axes[5].set_title("Policy entropy (not logged)")
axes[5].set_xlabel("training iteration")
axes[5].set_ylabel("entropy")
axes[5].grid(alpha=0.3)

plt.tight_layout()
curves_path = AGENT_DIR / "learning_curves.png"
fig.savefig(curves_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved {curves_path}")

# --- Dedicated win rate graph --------------------------------------------
fig_w, ax_w = plt.subplots(figsize=(7, 4))
if win_rate_col is not None:
    ax_w.plot(x, metrics_df[win_rate_col], color="#2a9d8f", linewidth=2)
    ax_w.axhline(
        args.win_rate_threshold, color="red", linestyle="--",
        label=f"threshold ({args.win_rate_threshold})",
    )
    ax_w.legend(loc="lower right")
ax_w.set_title("Win rate over training")
ax_w.set_xlabel("training iteration")
ax_w.set_ylabel("win rate")
ax_w.set_ylim(-0.05, 1.05)
ax_w.grid(alpha=0.3)
plt.tight_layout()
win_rate_path = AGENT_DIR / "win_rate.png"
fig_w.savefig(win_rate_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved {win_rate_path}")

# --- Dedicated reward graph ----------------------------------------------
fig_r, ax_r = plt.subplots(figsize=(7, 4))
if return_col is not None:
    ax_r.plot(x, metrics_df[return_col], color="#e76f51", linewidth=2)
ax_r.set_title("Reward (episode return) over training")
ax_r.set_xlabel("training iteration")
ax_r.set_ylabel("episode return")
ax_r.grid(alpha=0.3)
plt.tight_layout()
reward_path = AGENT_DIR / "reward.png"
fig_r.savefig(reward_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved {reward_path}")


### Evaluate vs Random baseline


In [ ]:
# Quantitative evaluation against a uniform-random baseline. Symmetric
# across player-0 / player-1 to account for Connect Four's first-move
# advantage.
NUM_EVAL_GAMES_PER_SIDE = 250

main_fn = functools.partial(greedy_action, main_module)
random_baseline = evaluate(
    main_fn, random_action, NUM_EVAL_GAMES_PER_SIDE, args.env,
)

print("Main vs Random baseline:")
for role, stats in random_baseline.items():
    print(
        f"  {role}: win={stats['win_rate']:.3f}  "
        f"loss={stats['loss_rate']:.3f}  draw={stats['draw_rate']:.3f}  "
        f"(n={stats['games']})"
    )

random_summary = overall_win_rate_ci(random_baseline)
print(
    f"  overall: win_rate={random_summary['win_rate']:.3f} "
    f"(95% CI [{random_summary['ci95_low']:.3f}, "
    f"{random_summary['ci95_high']:.3f}], n={random_summary['games']})"
)


### Evaluate vs league snapshots


In [ ]:
# Evaluate main vs every snapshot the SelfPlayCallback froze during
# training. Because the callback only snapshots main after win rate
# crosses the threshold, main should dominate its earlier selves — if
# it doesn't, that's a signal of cycling / catastrophic forgetting.
NUM_LEAGUE_GAMES_PER_SIDE = 50

league_eval = {}
if not snapshot_modules:
    print("No league snapshots were created during training — skipping.")
else:
    for name, mod in snapshot_modules.items():
        opp_fn = functools.partial(greedy_action, mod)
        league_eval[name] = evaluate(
            main_fn, opp_fn, NUM_LEAGUE_GAMES_PER_SIDE, args.env,
        )
        agg = overall_win_rate_ci(league_eval[name])
        p0 = league_eval[name]["main_as_p0"]["win_rate"]
        p1 = league_eval[name]["main_as_p1"]["win_rate"]
        print(
            f"main vs {name}: p0={p0:.2f} p1={p1:.2f} "
            f"overall={agg['win_rate']:.2f} "
            f"(95% CI [{agg['ci95_low']:.2f}, {agg['ci95_high']:.2f}], "
            f"n={agg['games']})"
        )


### Record games as GIFs


In [ ]:
# Record games as GIFs using Farama PettingZoo's Connect Four renderer.
#
# We can't feed PettingZoo's observations directly to the trained
# RLModule — the module was trained on OpenSpiel's `info_state` encoding
# and would silently mis-interpret PettingZoo's (6,7,2) observation
# tensor. Instead we drive two envs in lockstep:
#   - OpenSpiel (`Environment`) provides the observations + legal actions
#     the policy was trained on.
#   - PettingZoo (`connect_four_v3`) mirrors every action purely so we
#     can grab its rgb_array frames for the video.
# Because Connect Four is deterministic (single action deterministically
# drops into the lowest open cell of a column), mirroring is exact.
try:
    from pettingzoo.classic import connect_four_v3
    PETTINGZOO_AVAILABLE = True
except ImportError as e:
    print(f"PettingZoo not installed ({e}); skipping video recording.")
    print("Install with: !pip install 'pettingzoo[classic]'")
    PETTINGZOO_AVAILABLE = False


def record_game_gif(env_name, agent_p0, agent_p1, output_path):
    """Play one game and save its PettingZoo-rendered frames as a GIF."""
    pz_env = connect_four_v3.env(render_mode="rgb_array")
    pz_env.reset(seed=0)

    osp_env = Environment(env_name)
    time_step = osp_env.reset()

    frames = [pz_env.render()]
    while not time_step.last():
        pid = time_step.observations["current_player"]
        obs = np.asarray(
            time_step.observations["info_state"][pid], dtype=np.float32,
        )
        legal = time_step.observations["legal_actions"][pid]
        action = (agent_p0 if pid == 0 else agent_p1)(obs, legal)

        # Mirror the action into PettingZoo so we can render it, then
        # step OpenSpiel so the next observation is ready for the policy.
        pz_env.step(action)
        time_step = osp_env.step([action])
        frames.append(pz_env.render())

    pz_env.close()

    rewards = time_step.rewards
    if rewards[0] > 0:
        result = "p0 wins"
    elif rewards[1] > 0:
        result = "p1 wins"
    else:
        result = "draw"

    # Hold the final frame for a few extra ticks so viewers can see the
    # outcome before the GIF loops.
    frames.extend([frames[-1]] * 4)

    imageio.mimsave(str(output_path), frames, fps=1.5, loop=0)
    return {
        "moves": len(frames) - 5,  # minus the 1 initial + 4 held frames
        "result": result,
        "path": str(output_path),
        "num_frames": len(frames),
    }


recorded_games = {}
if PETTINGZOO_AVAILABLE:
    # Record one game with main on each side vs random, and (if the
    # league has any snapshots) one game vs the earliest snapshot.
    video_specs = [
        ("main_vs_random_p0.gif", main_fn, random_action),
        ("main_vs_random_p1.gif", random_action, main_fn),
    ]
    if snapshot_modules:
        first_snap_name = next(iter(snapshot_modules))
        first_snap_fn = functools.partial(
            greedy_action, snapshot_modules[first_snap_name],
        )
        video_specs.append((
            f"main_vs_{first_snap_name}.gif",
            main_fn,
            first_snap_fn,
        ))

    for filename, a0, a1 in video_specs:
        out_path = VIDEO_DIR / filename
        info = record_game_gif(args.env, a0, a1, out_path)
        recorded_games[filename] = info
        print(f"Saved {out_path.name}: {info['moves']} moves, {info['result']}")

    print(f"\nAll videos saved to: {VIDEO_DIR}")


### Persist artifacts


In [ ]:
# Copy the final checkpoint to ARTIFACT_DIR/checkpoint so it survives
# across Colab runtimes (on Google Drive when USE_GDRIVE is enabled),
# and write a JSON summary so the README numbers are generated by the
# notebook rather than hand-typed.
checkpoint_src = Path(final_checkpoint.path)
checkpoint_dst = AGENT_DIR / "checkpoint"
if checkpoint_dst.exists():
    shutil.rmtree(checkpoint_dst)
shutil.copytree(checkpoint_src, checkpoint_dst)
print(f"Checkpoint copied to: {checkpoint_dst}")

last_result = results.trials[0].last_result or {}
final_iter = int(
    last_result.get(TRAINING_ITERATION, metrics_df["training_iteration"].iloc[-1])
)
final_steps = int(
    last_result.get(
        NUM_ENV_STEPS_SAMPLED_LIFETIME,
        metrics_df.get(NUM_ENV_STEPS_SAMPLED_LIFETIME, pd.Series([0])).iloc[-1],
    )
)
final_league = int(
    last_result.get(
        "league_size",
        metrics_df.get("league_size", pd.Series([0])).iloc[-1],
    )
)
wall_clock = float(last_result.get("time_total_s", 0.0))

summary = {
    "algo": args.algo,
    "env": args.env,
    "training_iterations": final_iter,
    "total_env_steps": final_steps,
    "league_size": final_league,
    "wall_clock_seconds": wall_clock,
    "artifact_dir": str(ARTIFACT_DIR),
    "agent_dir": str(AGENT_DIR),
    "drive_mounted": drive_mounted,
    "eval_vs_random": {
        "per_side": random_baseline,
        "overall": random_summary,
    },
    "eval_vs_league": {
        name: {
            "per_side": stats,
            "overall": overall_win_rate_ci(stats),
        }
        for name, stats in league_eval.items()
    },
    "checkpoint_path": str(checkpoint_dst),
    "graphs": {
        "learning_curves": str(AGENT_DIR / "learning_curves.png"),
        "win_rate": str(AGENT_DIR / "win_rate.png"),
        "reward": str(AGENT_DIR / "reward.png"),
    },
    "videos": recorded_games,
}

results_json_path = AGENT_DIR / "results.json"
with open(results_json_path, "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(f"Summary written to: {results_json_path}")
print(json.dumps(summary, indent=2, default=str))


### Export agent for tournament

In [ ]:
# Export this agent for the tournament notebook. We save:
#   - rl_module/   the standalone main RLModule (lighter to load than a
#                  full Algorithm; used by adapter.py at tournament time)
#   - adapter.py   load_agent(agent_dir, env_name) -> act callable
#   - meta.json    algo + training metadata so the tournament notebook
#                  can label rows / sort by recency
# The full Algorithm checkpoint already lives under AGENT_DIR/checkpoint/
# from the previous cell, so adapters that need league snapshots can
# still reach for it.
import textwrap

rl_module_dir = AGENT_DIR / "rl_module"
if rl_module_dir.exists():
    shutil.rmtree(rl_module_dir)
rl_module_dir.mkdir(parents=True, exist_ok=True)
main_module.save_to_path(str(rl_module_dir))
print(f"Saved standalone RLModule to: {rl_module_dir}")

adapter_src = textwrap.dedent('''\
    """Tournament adapter for an RLlib-trained agent.

    Loaded by the tournament notebook via importlib. Exposes a single
    ``load_agent(agent_dir, env_name)`` function that returns a callable
    matching the contract:

        act(obs: np.ndarray, legal_actions: list[int], state=None) -> int

    ``state`` is accepted for compatibility with MCTS-style agents
    (AlphaZero); RLlib net-based agents ignore it.
    """

    from pathlib import Path

    import numpy as np
    import torch


    def _load_module(agent_dir):
        """Load just the trained RLModule, no Ray actors required."""
        from ray.rllib.core.rl_module.rl_module import RLModule

        rl_module_dir = Path(agent_dir) / "rl_module"
        return RLModule.from_checkpoint(str(rl_module_dir))


    def load_agent(agent_dir, env_name):  # noqa: ARG001 (env_name unused here)
        module = _load_module(agent_dir)

        def act(obs, legal_actions, state=None):  # noqa: ARG001
            with torch.no_grad():
                out = module.forward_inference(
                    {"obs": torch.from_numpy(np.asarray(obs)).unsqueeze(0)}
                )

            # Different RLModule families expose action selection through
            # different keys. Dispatch on what's present so this single
            # adapter covers both PPO (action_dist_inputs) and DQN
            # (q_values), and falls back to greedy "actions" if neither
            # is exposed.
            if "action_dist_inputs" in out:
                scores = out["action_dist_inputs"][0].detach().cpu().numpy()
            elif "q_values" in out:
                scores = out["q_values"][0].detach().cpu().numpy()
            elif "actions" in out:
                a = int(out["actions"][0].item())
                if a in legal_actions:
                    return a
                return int(np.random.choice(legal_actions))
            else:
                raise RuntimeError(
                    f"Adapter does not know how to read RLModule output "
                    f"with keys {list(out.keys())}"
                )

            masked = np.full_like(scores, -np.inf, dtype=np.float32)
            masked[legal_actions] = scores[legal_actions]
            return int(np.argmax(masked))

        return act
''')
adapter_path = AGENT_DIR / "adapter.py"
adapter_path.write_text(adapter_src)
print(f"Wrote adapter to: {adapter_path}")

meta = {
    "agent_name": AGENT_NAME,
    "algo": args.algo,
    "env": args.env,
    "training_iterations": final_iter,
    "total_env_steps": final_steps,
    "league_size": final_league,
    "wall_clock_seconds": wall_clock,
    "in_training_win_rate": (
        float(metrics_df[win_rate_col].iloc[-1])
        if win_rate_col is not None else None
    ),
    "win_rate_threshold": args.win_rate_threshold,
    "framework": args.framework,
    "rl_module_path": str(rl_module_dir),
    "checkpoint_path": str(checkpoint_dst),
    "results_json": str(results_json_path),
}
meta_path = AGENT_DIR / "meta.json"
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2, default=str)
print(f"Wrote meta to: {meta_path}")


## Play vs the trained agent (interactive, optional)

Set `args.num_episodes_human_play > 0` and re-run the cell below to play
on the command line. Skipped by default so the notebook can run headless.


In [ ]:
# Restore the trained Algorithm and play against a human on the command line.
# This cell is a no-op by default (args.num_episodes_human_play == 0) so the
# notebook can be executed end-to-end headlessly (e.g. in CI / "Run all").
# The eval_algo built above is reused so we don't rebuild from scratch.
if args.num_episodes_human_play > 0:
    if "eval_algo" not in globals():
        from ray.rllib.algorithms.algorithm import Algorithm
        eval_algo = Algorithm.from_checkpoint(str(Path(final_checkpoint.path)))

    rl_module = eval_algo.get_module("main")

    # Play from the command line against the trained agent in an actual
    # (non-RLlib-wrapped) open-spiel env.
    human_player = 1
    env = Environment(args.env)
    num_episodes = 0

    while num_episodes < args.num_episodes_human_play:
        print("You play as {}".format("o" if human_player else "x"))
        time_step = env.reset()
        while not time_step.last():
            player_id = time_step.observations["current_player"]
            if player_id == human_player:
                action = ask_user_for_action(time_step)
            else:
                obs = np.asarray(
                    time_step.observations["info_state"][player_id],
                    dtype=np.float32,
                )
                legal = time_step.observations["legal_actions"][player_id]
                logits = (
                    rl_module.forward_inference(
                        {"obs": torch.from_numpy(obs).unsqueeze(0)}
                    )["action_dist_inputs"][0]
                    .detach()
                    .numpy()
                )
                # Mask illegal moves BEFORE argmax so we never pick one
                # and then have to fall back to a random legal move.
                masked = np.full_like(logits, -np.inf, dtype=np.float32)
                masked[legal] = logits[legal]
                action = int(np.argmax(masked))
            time_step = env.step([action])
            print(f"\n{env.get_state}")

        print(f"\n{env.get_state}")

        print("End of game!")
        if time_step.rewards[human_player] > 0:
            print("You win")
        elif time_step.rewards[human_player] < 0:
            print("You lose")
        else:
            print("Draw")
        # Switch order of players.
        human_player = 1 - human_player
        num_episodes += 1


In [ ]:
# Shut down the eval algorithm and its actors.
eval_algo.stop()
